## Final dataset for training

Crystal Tubbs wrote this notebook as the second half of the scientific dataset work. `00_build_scientific_dataset.ipynb` does the first pass on metadata; here we cut interviews into **30-second** clips, re-check splits, and write `train_dm.csv`, `valid_dm.csv`, and `test_dm.csv` for `02_train_scientific_baseline.ipynb`.

Training clips use a **20 s stride** (overlap); val and test use a **30 s stride** so evaluation windows do not overlap. Later cells check for speaker leakage, class balance, duplicate `(path, clip_idx)` rows, and a small random sample of decoded durations.


Google Drive (Colab): mount it, then the next cell `cd`s into this repo so `import config` works.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/NeuralNets_Project_DementiaNet

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


The `%cd` line should leave you in the repo root (the folder that contains `config.py`). If you are not on Colab, `cd` there yourself before importing.


In [ ]:
from config import *
import pandas as pd
import numpy as np
import torch
import torchaudio
import random
import re
from pathlib import Path

OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


Seeds for `random`, NumPy, and PyTorch are set in the first code cell so subsampling and anything else stochastic stays repeatable.


Load Metadata

Load the original metadata files for dementia and non-dementia speakers, which contain speaker information and dataset split hints used to guide dataset construction.


In [ ]:
dementia_meta = pd.read_csv(DEMENTIA_META)
nodementia_meta = pd.read_csv(NODEMENTIA_META)

print("Dementia metadata shape:", dementia_meta.shape)
print("No dementia metadata shape:", nodementia_meta.shape)

dementia_meta.head()

Dementia metadata shape: (84, 16)
No dementia metadata shape: (61, 10)


,name,dementia type,birth,death,first symptoms,URLs after symptoms,5 years,5 < 10 years,10 < 15 years,gender,ethnicity,datasplit,language,unknown 1,unkown 2,unknown 3
0,Abe Burrows,Alzheimer,1910,1985,1975.0,NaN,https://www.youtube.com/watch?v=VezbsmCriw4,NaN,NaN,male,Caucasian/White,train,NaN,NaN,NaN,NaN
1,Aileen Hernandez,Dementia,1926,2017,2012.0,https://youtu.be/x7hujcEhQuY,https://youtu.be/CshhDl-YwkY \nhttps://youtu.b...,NaN,NaN,female,Black/African American,train,NaN,NaN,NaN,NaN
2,Alan Ramsey,Dementia,1938,2020,2015.0,NaN,https://www.youtube.com/watch?v=CHeXE4c6EDI,NaN,NaN,male,Caucasian/White,train,NaN,NaN,NaN,NaN
3,Allan Burns,Lewy body,1935,2021,NaN,NaN,https://www.youtube.com/watch?v=aD3hL-kWoPc,NaN,NaN,male,Caucasian/White,train,NaN,NaN,NaN,NaN
4,Andrew Sachs,Dementia,1930,2016,2012.0,NaN,NaN,https://youtu.be/FSgKLooW1LM,https://youtu.be/3V1iFmavqG4,male,NaN,train,NaN,NaN,NaN,NaN


Normalize Speaker Names

Standardize speaker names by converting them to a consistent format so that names in the metadata match the corresponding audio folder names.


In [ ]:
def normalize(name):
    name = str(name).lower()
    name = re.sub(r"[^\w]", "", name)
    return name

dementia_meta["speaker"] = dementia_meta["name"].apply(normalize)
nodementia_meta["speaker"] = nodementia_meta["name"].apply(normalize)

dementia_meta["label"] = "dementia"
nodementia_meta["label"] = "nodementia"

Audio Clip Segmentation Strategy

To ensure consistent input length for the speech classification model, recordings are divided into fixed length 30 second clips. This segmentation allows recordings of different durations to be processed uniformly while also increasing the number of usable training samples.

Two segmentation strategies are used depending on the dataset split. For the training set, clips are generated with a stride of 20 seconds, which introduces partial overlap between adjacent segments. This overlap increases the diversity of training examples by allowing the model to observe slightly different contextual windows from the same recording.

For the validation and test sets, clips are generated using a stride of 30 seconds, producing non overlapping segments. This ensures that evaluation is performed on distinct portions of audio without repeated or overlapping content, resulting in a cleaner and more reliable assessment of model performance.

All segmentation occurs after the dataset has already been split by speaker. As a result, every clip produced from a recording remains within the same dataset partition, preserving speaker independence across training, validation, and test sets.


Convert Recordings to Multiple 30-Second Clips

This step implements the segmentation strategy described above. Each recording is divided into fixed 30 second clips. For validation and test datasets, clips are generated using a stride of 30 seconds, producing non overlapping segments. For example, a 90 second recording produces three clips beginning at 0 seconds, 30 seconds, and 60 seconds.

For the training dataset, a stride of 20 seconds is used. This introduces approximately 33 percent overlap between clips, allowing each new clip to contain at least 20 seconds of new speech. The overlapping strategy increases the number of training samples and exposes the model to slightly different contextual windows from the same recording.

Recordings shorter than 30 seconds produce a single clip starting at 0 seconds and are padded during feature extraction if necessary.

Using fixed length clips standardizes model input size and provides sufficient context to capture cognitive and linguistic markers associated with dementia, such as pauses, topic drift, and reduced vocabulary diversity.

Dataset splitting is performed earlier in the pipeline at the speaker level. This ensures that recordings from the same speaker appear in only one dataset partition. After segmentation, all clips generated from a recording remain within their assigned dataset split, preserving speaker separation and preventing speaker leakage.


In [ ]:
def expand_clips(row, stride: float):
    """Slice one interview into fixed-length clip rows.

    CLIP_SECONDS is from config; callers pass stride (TRAIN_STRIDE or EVAL_STRIDE).
    """
    duration = row["duration_sec"]
    clips = []
    start = 0.0
    idx = 0
    while start + CLIP_SECONDS <= duration + 1e-3:
        clips.append({
            **row.to_dict(),
            "start_sec": float(start),
            "clip_sec": float(CLIP_SECONDS),
            "clip_idx": idx,
        })
        start += stride
        idx += 1
    if not clips:
        clips.append({
            **row.to_dict(),
            "start_sec": 0.0,
            "clip_sec": float(CLIP_SECONDS),
            "clip_idx": 0,
        })
    return clips


Combine Metadata

Merge the dementia and non-dementia metadata tables into a single dataframe so all speakers can be processed together through a unified pipeline.


In [ ]:
metadata = pd.concat([dementia_meta, nodementia_meta], ignore_index=True)

print(metadata.columns)
metadata.head()

Index(['name', 'dementia type', 'birth', 'death', 'first symptoms',
       'URLs after symptoms', '5 years', '5 < 10 years', '10 < 15 years',
       'gender', 'ethnicity', 'datasplit', 'language', 'unknown 1', 'unkown 2',
       'unknown 3', 'speaker', 'label', 'birthdate', 'deathdate',
       'age above 70', 'age above 55 less than 70', 'age less than 55'],
      dtype='object')


,name,dementia type,birth,death,first symptoms,URLs after symptoms,5 years,5 < 10 years,10 < 15 years,gender,...,unknown 1,unkown 2,unknown 3,speaker,label,birthdate,deathdate,age above 70,age above 55 less than 70,age less than 55
0,Abe Burrows,Alzheimer,1910.0,1985,1975.0,NaN,https://www.youtube.com/watch?v=VezbsmCriw4,NaN,NaN,male,...,NaN,NaN,NaN,abeburrows,dementia,NaN,NaN,NaN,NaN,NaN
1,Aileen Hernandez,Dementia,1926.0,2017,2012.0,https://youtu.be/x7hujcEhQuY,https://youtu.be/CshhDl-YwkY \nhttps://youtu.b...,NaN,NaN,female,...,NaN,NaN,NaN,aileenhernandez,dementia,NaN,NaN,NaN,NaN,NaN
2,Alan Ramsey,Dementia,1938.0,2020,2015.0,NaN,https://www.youtube.com/watch?v=CHeXE4c6EDI,NaN,NaN,male,...,NaN,NaN,NaN,alanramsey,dementia,NaN,NaN,NaN,NaN,NaN
3,Allan Burns,Lewy body,1935.0,2021,NaN,NaN,https://www.youtube.com/watch?v=aD3hL-kWoPc,NaN,NaN,male,...,NaN,NaN,NaN,allanburns,dementia,NaN,NaN,NaN,NaN,NaN
4,Andrew Sachs,Dementia,1930.0,2016,2012.0,NaN,NaN,https://youtu.be/FSgKLooW1LM,https://youtu.be/3V1iFmavqG4,male,...,NaN,NaN,NaN,andrewsachs,dementia,NaN,NaN,NaN,NaN,NaN


Build Speaker Split Lists

Create speaker level sets for the train, validation, and test partitions using the datasplit field in the metadata. These sets ensure that recordings from the same speaker remain within a single dataset split.


In [ ]:
train_speakers = set(metadata[metadata["datasplit"] == "train"]["speaker"])
valid_speakers = set(metadata[metadata["datasplit"] == "valid"]["speaker"])
test_speakers = set(metadata[metadata["datasplit"] == "test"]["speaker"])

print("train speakers:", len(train_speakers))
print("valid speakers:", len(valid_speakers))
print("test speakers:", len(test_speakers))

train speakers: 118
valid speakers: 25
test speakers: 2


Fix Broken Test Split

The original metadata test split contained too few speakers and did not provide reliable class representation with only 2 counts of 'test' in Dementia and none in NoDementia.

The test speakers are first merged into the validation pool, and a new balanced test set is sampled from the training speakers so that both dementia and nondementia classes are represented.


In [ ]:
valid_speakers = valid_speakers.union(test_speakers)
test_speakers = set()

train_meta = metadata[metadata["speaker"].isin(train_speakers)]

dementia_train = train_meta[train_meta["label"] == "dementia"]["speaker"].unique()
nodementia_train = train_meta[train_meta["label"] == "nodementia"]["speaker"].unique()

# sample approximately 15% of speakers per class to form a balanced test set
n_test = int(min(len(dementia_train), len(nodementia_train)) * TEST_SPLIT_PERCENT)

test_dementia = random.sample(list(dementia_train), n_test)
test_nodementia = random.sample(list(nodementia_train), n_test)

test_speakers = set(test_dementia) | set(test_nodementia)

train_speakers = train_speakers - test_speakers

Scan Audio Files

Search the audio directories for .wav files, verify that each file can be loaded successfully, and collect basic recording metadata such as speaker ID, file path, label, and duration.


In [ ]:
rows = []
invalid_files = []


def process_folder(folder, label):
    for wav in folder.glob("**/*.wav"):
        speaker = normalize(wav.parent.name)
        try:
            waveform, sr = torchaudio.load(wav)
            duration_sec = waveform.shape[1] / sr
        except Exception as e:
            invalid_files.append((str(wav), str(e)))
            continue
        rows.append({
            "file": wav.name,
            "label": label,
            "path": str(wav),
            "speaker": speaker,
            "duration_sec": duration_sec,
        })


process_folder(DEMENTIA_DIR, "dementia")
process_folder(NODEMENTIA_DIR, "nodementia")

df = pd.DataFrame(rows)
print("Total valid audio files:", len(df))
if invalid_files:
    print(f"Skipped {len(invalid_files)} unreadable file(s):")
    for path, err in invalid_files:
        print(f"  {path}: {err}")
else:
    print("No invalid files found.")


Total valid audio files: 369
No invalid files found.


Assign Dataset Split

Assign each recording to the train, validation, or test partition based on speaker identity so that recordings from the same speaker never appear in multiple splits.


In [ ]:
def assign_split(speaker):
    if speaker in train_speakers:
        return "train"
    if speaker in valid_speakers:
        return "valid"
    if speaker in test_speakers:
        return "test"
    return None

print("DF columns:", df.columns)
print("Number of rows (recordings):", len(df))
print(df.head())

df["split"] = df["speaker"].apply(assign_split)
df = df.dropna(subset=["split"])

# Expand clips:
# Training -> 20s stride (~33% overlap) for additional training samples
# Validation/Test -> 30s stride (non overlapping) for clean evaluation
expanded_rows = []
for _, row in df.iterrows():
    stride = TRAIN_STRIDE if row["split"] == "train" else EVAL_STRIDE
    expanded_rows.extend(expand_clips(row, stride=stride))

df = pd.DataFrame(expanded_rows).reset_index(drop=True)

print("\nAfter expansion (clips):", len(df))
print(df[["file", "duration_sec", "start_sec", "clip_sec", "clip_idx"]].head(10))

DF columns: Index(['file', 'label', 'path', 'speaker', 'duration_sec'], dtype='object')
Number of rows (recordings): 369
                      file     label  \
0         jimmclean_10.wav  dementia   
1  annettemichelson_15.wav  dementia   
2        alanramsey_10.wav  dementia   
3        georgerobb_10.wav  dementia   
4       andrewsachs_15.wav  dementia   

                                                path           speaker  \
0  /content/drive/My Drive/CS7357_Project/data/de...         jimmclean   
1  /content/drive/My Drive/CS7357_Project/data/de...  annettemichelson   
2  /content/drive/My Drive/CS7357_Project/data/de...        alanramsey   
3  /content/drive/My Drive/CS7357_Project/data/de...        georgerobb   
4  /content/drive/My Drive/CS7357_Project/data/de...       andrewsachs   

   duration_sec  
0          90.0  
1          90.0  
2          55.0  
3          22.0  
4          90.0  

After expansion (clips): 443
                      file  duration_sec  start_sec  cl

Clip Expansion Summary

After segmentation, the dataset grows from the original recordings to a larger set of fixed length clips. Longer recordings contribute multiple clips, while shorter recordings produce a single padded clip. Training recordings may produce overlapping clips due to the smaller stride used for data augmentation.


Create Manifest Tables

Create structured dataset manifests for the train, validation, and test splits containing file paths and labels used during model training.


In [ ]:
train_df = df[df["split"] == "train"]
valid_df = df[df["split"] == "valid"]
test_df = df[df["split"] == "test"]

print("train:", len(train_df))
print("valid:", len(valid_df))
print("test:", len(test_df))

train: 325
valid: 80
test: 38


Verify Class Balance

Check the distribution of dementia and non-dementia samples in each split to confirm that both classes are represented.


In [ ]:
print("TRAIN")
print(train_df["label"].value_counts())

print("\nVALID")
print(valid_df["label"].value_counts())

print("\nTEST")
print(test_df["label"].value_counts())

TRAIN
label
dementia      202
nodementia    123
Name: count, dtype: int64

VALID
label
dementia      44
nodementia    36
Name: count, dtype: int64

TEST
label
nodementia    24
dementia      14
Name: count, dtype: int64


Verify Source Audio Counts

Confirm the number of dementia and nondementia recordings available in the dataset directories before building the final manifests


In [ ]:
print("dementia wav files:", len(list(DEMENTIA_DIR.glob("**/*.wav"))))
print("nodementia wav files:", len(list(NODEMENTIA_DIR.glob("**/*.wav"))))



dementia wav files: 131
nodementia wav files: 238


Verify No Speaker Leakage

Ensure that no speaker appears in more than one dataset split to prevent the model from learning speaker identity instead of the target task.


In [ ]:
train_set = set(train_df["speaker"])
valid_set = set(valid_df["speaker"])
test_set = set(test_df["speaker"])

print("train ∩ valid:", train_set & valid_set)
print("train ∩ test:", train_set & test_set)
print("valid ∩ test:", valid_set & test_set)

train ∩ valid: set()
train ∩ test: set()
valid ∩ test: set()


Save Final Dataset

Save the processed train, validation, and test manifests so the same dataset splits can be reused consistently during training and evaluation.


In [ ]:
manifest_cols = ["file", "label", "path", "speaker", "duration_sec", "start_sec", "clip_sec", "clip_idx"]

train_df[manifest_cols].to_csv(
    OUTPUT_DIR / "train_dm.csv",
    sep="\t",
    index=False
)

valid_df[manifest_cols].to_csv(
    OUTPUT_DIR / "valid_dm.csv",
    sep="\t",
    index=False
)

test_df[manifest_cols].to_csv(
    OUTPUT_DIR / "test_dm.csv",
    sep="\t",
    index=False
)

Quick eyeball on a few rows before writing manifests: you want duration_sec for the whole interview, start_sec and clip_sec for the 30 s window, and clip_idx stepping along each file.


In [ ]:
print(train_df[["file", "duration_sec", "start_sec", "clip_sec"]].head())
print(valid_df[["file", "duration_sec", "start_sec", "clip_sec"]].head())
print(test_df[["file", "duration_sec", "start_sec", "clip_sec"]].head())

                  file  duration_sec  start_sec  clip_sec
6    alanramsey_10.wav          55.0        0.0      30.0
7    alanramsey_10.wav          55.0       20.0      30.0
8    georgerobb_10.wav          22.0        0.0      30.0
9   andrewsachs_15.wav          90.0        0.0      30.0
10  andrewsachs_15.wav          90.0       20.0      30.0
                       file  duration_sec  start_sec  clip_sec
3   annettemichelson_15.wav          90.0        0.0      30.0
4   annettemichelson_15.wav          90.0       30.0      30.0
5   annettemichelson_15.wav          90.0       60.0      30.0
79            bbking_10.wav          90.0        0.0      30.0
80            bbking_10.wav          90.0       30.0      30.0
                      file  duration_sec  start_sec  clip_sec
0         jimmclean_10.wav     90.000000        0.0      30.0
1         jimmclean_10.wav     90.000000       30.0      30.0
2         jimmclean_10.wav     90.000000       60.0      30.0
28  maureenforrester_5.wav

Reload Dataset Manifests

Reload the saved train, validation, and test manifest files to confirm they were written correctly.

This verification step ensures the saved files match the in memory dataset and can be reliably used during model training and evaluation.


In [ ]:
train = pd.read_csv(TRAIN_MANIFEST, sep="\t")
valid = pd.read_csv(VALID_MANIFEST, sep="\t")
test = pd.read_csv(TEST_MANIFEST, sep="\t")

print("train:", len(train))
print("valid:", len(valid))
print("test:", len(test))


train: 325
valid: 80
test: 38


Verify Speaker Counts Per Split

This step computes the number of unique speakers present in each dataset split.

Compute the number of unique speakers in each dataset split. Speaker identities are extracted from file names, and verifying these counts ensures that recordings are distributed across multiple individuals and that the dataset structure is correct.


In [ ]:
def speaker_from_file(f):
    return Path(f).stem.split("_")[0]


print("train speakers:", train["file"].apply(speaker_from_file).nunique())
print("valid speakers:", valid["file"].apply(speaker_from_file).nunique())
print("test speakers:", test["file"].apply(speaker_from_file).nunique())


train speakers: 97
valid speakers: 26
test speakers: 13


Analyze Clip Distribution Per Speaker

Examine how many clips each speaker contributes to the dataset.

 This analysis helps ensure the dataset is not dominated by a small number of speakers, reducing the risk that the model learns speaker specific characteristics instead of dementia related speech patterns.


In [ ]:
train["speaker"] = train["file"].apply(speaker_from_file)
valid["speaker"] = valid["file"].apply(speaker_from_file)
test["speaker"] = test["file"].apply(speaker_from_file)

print("Train clips per speaker")
print(train.groupby("speaker").size().describe())

print("\nValid clips per speaker")
print(valid.groupby("speaker").size().describe())

print("\nTest clips per speaker")
print(test.groupby("speaker").size().describe())

Train clips per speaker
count    97.000000
mean      3.350515
std       2.385028
min       1.000000
25%       1.000000
50%       3.000000
75%       5.000000
max      12.000000
dtype: float64

Valid clips per speaker
count    26.000000
mean      3.076923
std       1.695242
min       1.000000
25%       2.000000
50%       3.000000
75%       3.750000
max       7.000000
dtype: float64

Test clips per speaker
count    13.000000
mean      2.923077
std       1.605280
min       1.000000
25%       2.000000
50%       3.000000
75%       3.000000
max       6.000000
dtype: float64


Check Label Consistency Per Speaker

Verify that each speaker is associated with only one class label. If a speaker appeared with multiple labels, it would indicate a metadata or labeling error and could introduce ambiguous supervision during training.


In [ ]:
all_df = pd.concat([train, valid, test])

speaker_labels = all_df.groupby("speaker")["label"].nunique()

print("Speakers with multiple labels:", (speaker_labels > 1).sum())


Speakers with multiple labels: 0


Inspect Audio Duration Distribution

Sample recordings from the dataset and compute summary statistics for their durations. This analysis provides insight into recording length variability and helps justify the use of fixed length clip segmentation during training.


Look for Duplicates

Check for duplicate clip entries. Because multiple clips can originate from the same recording, duplicates are evaluated using (path, clip_idx) pairs rather than file paths alone. Each source file and clip index combination should appear only once.


In [ ]:
# Clips from the same file are intentional — check for duplicate (path, clip_idx) pairs instead
print(train[["path", "clip_idx"]].duplicated().sum())
print(valid[["path", "clip_idx"]].duplicated().sum())
print(test[["path", "clip_idx"]].duplicated().sum())

0
0
0


Dataset Summary Statistics

Provide a high level summary of the constructed dataset including the total number of clips, the number of unique speakers, and the distribution of class labels.

These summary values provide a final confirmation that the dataset contains both dementia and non dementia samples and that the data is distributed across many speakers.


In [ ]:
print("Total clips:", len(all_df))
print("Total speakers:", all_df["speaker"].nunique())

print("\nClips per class:")
print(all_df["label"].value_counts())


Total clips: 443
Total speakers: 136

Clips per class:
label
dementia      260
nodementia    183
Name: count, dtype: int64


Compare Duration Distributions Across Splits

Compare recording duration statistics across the training, validation, and test splits. Similar distributions indicate that evaluation data reflects the same recording characteristics seen during training.


In [ ]:
def get_duration(path):
    audio, sr = torchaudio.load(path)
    return audio.shape[1] / sr


n_sample = min(40, len(train), len(valid), len(test))
train_durations = train["path"].sample(n=min(n_sample, len(train)), random_state=RANDOM_SEED).apply(get_duration)
valid_durations = valid["path"].sample(n=min(n_sample, len(valid)), random_state=RANDOM_SEED).apply(get_duration)
test_durations = test["path"].sample(n=min(n_sample, len(test)), random_state=RANDOM_SEED).apply(get_duration)

print("Train decoded duration (s)")
print(train_durations.describe())
print("\nValid decoded duration (s)")
print(valid_durations.describe())
print("\nTest decoded duration (s)")
print(test_durations.describe())


Train duration stats
count     30.000000
mean      67.535015
std       30.195488
min       15.007438
25%       40.250000
50%       72.000000
75%       90.000000
max      162.003719
Name: path, dtype: float64

Valid duration stats
count     30.000000
mean      76.670041
std       25.014167
min       14.014512
25%       54.250000
50%       90.000000
75%       90.000000
max      110.001270
Name: path, dtype: float64

Test duration stats
count    30.000000
mean     74.102259
std      19.232789
min      28.000000
25%      64.500000
50%      79.000000
75%      90.000000
max      90.016236
Name: path, dtype: float64


## What we checked

Speakers sit in exactly one split, labels do not contradict per speaker, train windows overlap with a 20 s stride while val/test use a 30 s stride, and duplicate `(path, clip_idx)` pairs are zero. Rough clip counts end up on the order of a few hundred train rows and smaller val/test sets depending on the source release.
